In [1]:
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [2]:
import os
from pyspark.sql import SparkSession
from google.colab import drive
import re
from pyspark.sql import functions as F

In [3]:
spark = (SparkSession.builder
         .appName("HospitalCharges")
         .getOrCreate())

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
# Reading the data
hospital_data = spark.read.parquet("/content/drive/My Drive/ProjectBigData/01 datasets/complete-data/hospital_charges_parquet30_cleaned")

In [7]:
hospital_data.show()

+--------------------+-----------+-----------+--------+-----------+------+----------+---------------------+-------------------------------+--------------------+--------------------+-------------+
|         description|     code_1|code_1_type|  code_2|code_2_type|code_3|   setting|standard_charge_gross|standard_charge_discounted_cash|          payer_name|           plan_name|hospital_name|
+--------------------+-----------+-----------+--------+-----------+------+----------+---------------------+-------------------------------+--------------------+--------------------+-------------+
|colonoscopy stoma...|PX-75000411|        cdm|   44406|        cpt|  0750| inpatient|              2782.00|                        2086.50|medicare map alte...|medicare map alte...|massachusetts|
|gft skin sub f/s ...|PX-76000183|        cdm|   15278|        cpt|  0761|outpatient|               418.00|                         313.50|international com...|seguros reservas ...|massachusetts|
|tenex procedure w..

In [8]:
hospital_data.count()

13598308

In [9]:
hospital_data.columns

['description',
 'code_1',
 'code_1_type',
 'code_2',
 'code_2_type',
 'code_3',
 'setting',
 'standard_charge_gross',
 'standard_charge_discounted_cash',
 'payer_name',
 'plan_name',
 'hospital_name']

### Identify if discounted cash varies across payer/plan

In [11]:
from pyspark.sql import functions as F

check_df = (
    hospital_data.groupBy(
        "hospital_name",
        "description",
        "code_1", "code_2", "code_3",
        "setting"
    )
    .agg(
        F.countDistinct("standard_charge_discounted_cash").alias("num_unique_cash_prices"),
        F.collect_set("standard_charge_discounted_cash").alias("cash_values")
    )
)

# Show groups where discounted cash price is NOT unique
check_df.filter(F.col("num_unique_cash_prices") > 1).show(truncate=False)


+-------------+----------------------------------------------------------------------+---------+--------+------+----------+----------------------+-------------------------------------------------+
|hospital_name|description                                                           |code_1   |code_2  |code_3|setting   |num_unique_cash_prices|cash_values                                      |
+-------------+----------------------------------------------------------------------+---------+--------+------+----------+----------------------+-------------------------------------------------+
|massachusetts|abortion with d&c, aspiration curettage or hysterotomy                |770      |NULL    |NULL  |inpatient |2                     |[21512.21, 21513.87]                             |
|massachusetts|acetaminophen 160 mg/5 ml oral liquid                                 |RX-100   |25009999|0250  |outpatient|8                     |[0.04, 0.31, 10.55, 0.13, 0.02, 0.41, 0.14, 0.03]|
|massachusetts|

Discounted cash price DOES vary across rows for the same procedure + hospital in dataset.

In [12]:
# dropping payer name and plan name

df_clean = hospital_data.drop("payer_name", "plan_name")


In [14]:
df_clean.select("code_2","code_2_type").show()

+--------+-----------+
|  code_2|code_2_type|
+--------+-----------+
|   44406|        cpt|
|   15278|        cpt|
|   25999|        cpt|
|   C1713|      hcpcs|
|   90656|      hcpcs|
|   93244|        cpt|
|   59830|        cpt|
|   L0460|      hcpcs|
|   35206|        cpt|
|25009999|      hcpcs|
|25009999|      hcpcs|
|   J9299|      hcpcs|
|25009999|      hcpcs|
|    NULL|    unknown|
|   35206|        cpt|
|   Q4132|      hcpcs|
|   C1713|      hcpcs|
|   C1768|      hcpcs|
|   11426|        cpt|
|   C1776|      hcpcs|
+--------+-----------+
only showing top 20 rows



In [16]:
# handling missing values

df_clean = (
    df_clean
    .withColumn("code_2_type", F.when(F.col("code_2_type").isNull(), "unknown").otherwise(F.col("code_2_type")))
    .withColumn("code_2", F.when(F.col("code_2").isNull(), "NO_CODE_2").otherwise(F.col("code_2")))
    .withColumn("code_3", F.when(F.col("code_3").isNull(), "NO_CODE_3").otherwise(F.col("code_3")))
)


### Deduplicate rows to get ONE cash price per procedure per hospital

dataset currently contains millions of duplicated rows per hospital-procedure combination,
because every payer-plan inflated the row count.

In [17]:
df_dedup = (
    df_clean
    .groupBy("hospital_name", "code_2", "setting")
    .agg(
        F.percentile_approx("standard_charge_discounted_cash", 0.5).alias("discounted_cash"),
        F.percentile_approx("standard_charge_gross", 0.5).alias("gross_price"),

        F.first("description").alias("description"),
        F.first("code_1").alias("code_1"),
        F.first("code_1_type").alias("code_1_type"),
        F.first("code_2_type").alias("code_2_type"),
        F.first("code_3").alias("code_3")
    )
)


In [19]:
df_dedup.count()

47245

Using median because hospitals often show:

- pharmacy-level duplicates

- departmental-level duplicates

- variation in subcharges

Median gives the representative price.

In [20]:
hospital_data.select("description").distinct().count()

27327

In [21]:
df_dedup.select("description").distinct().count()

10143

In [22]:
df_dedup.show(5)

+-------------+------+---------+---------------+-----------+--------------------+-----------+-----------+-----------+------+
|hospital_name|code_2|  setting|discounted_cash|gross_price|         description|     code_1|code_1_type|code_2_type|code_3|
+-------------+------+---------+---------------+-----------+--------------------+-----------+-----------+-----------+------+
|massachusetts| 0001U|inpatient|         987.75|     1317.0|rbc dna hea 35 ag...|PX-30003410|        cdm|        cpt|  0300|
|massachusetts| 0008U|inpatient|          381.0|      508.0|hpylori detection...|PX-31001079|        cdm|        cpt|  0310|
|massachusetts| 0027U|inpatient|        1301.25|     1735.0|jak2 exons 12 - 1...|PX-31000942|        cdm|        cpt|  0310|
|massachusetts| 0031A|inpatient|          56.25|       75.0|imm admn sarscov2...|PX-77100033|        cdm|        cpt|  0771|
|massachusetts| 0035U|inpatient|         4723.5|     6298.0|neuro csf detcj p...|PX-31001015|        cdm|        cpt|  0310|


In [23]:
df_dedup.select("hospital_name").distinct().count()

5

The original CMS price transparency dataset contained 27,327 unique procedure descriptions; however, this number reduced to 10,143 following the deduplication process. This reduction is both expected and methodologically appropriate. In the raw data, each clinical procedure is reported multiple times for every payer–plan combination, resulting in substantial textual duplication across descriptions that refer to the same underlying CPT or HCPCS code. Additionally, hospitals often publish department-specific or billing-unit variations of the same procedure, which appear as separate rows but do not represent distinct clinical services. By aggregating records at the level of hospital × primary procedure code × setting and extracting a single representative description for each group, the deduplication step eliminates redundant and noisy entries while preserving the true diversity of clinical procedures. The resulting 10,143 unique descriptions therefore represent the actual set of distinct services offered across hospitals, forming a cleaner, more accurate, and computationally efficient foundation for downstream machine learning and semantic search tasks.

### Analyzing target variable

In [24]:
df_dedup.select(
    F.min("discounted_cash"),
    F.expr("percentile_approx(discounted_cash, 0.25)").alias("Q1"),
    F.expr("percentile_approx(discounted_cash, 0.50)").alias("Median"),
    F.expr("percentile_approx(discounted_cash, 0.75)").alias("Q3"),
    F.max("discounted_cash")
).show()


+--------------------+-----+------+-------+--------------------+
|min(discounted_cash)|   Q1|Median|     Q3|max(discounted_cash)|
+--------------------+-----+------+-------+--------------------+
|                0.01|136.5| 562.5|2561.25|           7012500.0|
+--------------------+-----+------+-------+--------------------+



**Right Skewed**

In [25]:
# Relationship Between Cash Price and Gross Price

df_dedup.select(
    F.corr("gross_price", "discounted_cash").alias("corr_gross_cash")
).show()


+------------------+
|   corr_gross_cash|
+------------------+
|0.9999999870417638|
+------------------+



The discounted cash price exhibited a highly right-skewed distribution, with values ranging from $0.01 to more than $7 million, and with median and interquartile range values of $562.50 and $136.50–$2,561.25, respectively. This distribution reflects the inherent heterogeneity of hospital services, spanning low-cost pharmaceuticals to high-acuity inpatient surgical interventions. A Pearson correlation analysis between gross charges and discounted cash prices yielded a coefficient of 0.999999987, indicating an almost perfectly linear relationship. This confirms that discounted cash prices are predominantly derived as proportional adjustments to the gross charges, consistent with common hospital pricing practices for self-pay patients. As such, gross charge emerges as the single most influential predictor of discounted cash price. Furthermore, the extreme skewness of the distribution motivates the consideration of log-transformed targets during model development to improve numerical stability and predictive accuracy.

In [26]:
df_dedup.createOrReplaceTempView("tbl")
spark.sql("""
SELECT
  ROUND(gross_price/100)*100 AS price_bin,
  AVG(discounted_cash) AS avg_discounted_cash
FROM tbl
GROUP BY price_bin
ORDER BY price_bin
""").show(50)


+---------+-------------------+
|price_bin|avg_discounted_cash|
+---------+-------------------+
|     NULL|               NULL|
|      0.0| 16.408339653304434|
|    100.0|  75.93408422356399|
|    200.0| 146.99172087658602|
|    300.0| 224.24743678559068|
|    400.0|  299.1559911460895|
|    500.0|  372.6932860520094|
|    600.0|  449.2625754231051|
|    700.0|  523.9558828671329|
|    800.0|  598.0326147959184|
|    900.0|  670.8535059760957|
|   1000.0|  748.4296818181817|
|   1100.0|  826.0226984126983|
|   1200.0|  897.7308265582657|
|   1300.0|  975.0126403641883|
|   1400.0| 1048.1207120743034|
|   1500.0| 1126.4366783216783|
|   1600.0|  1197.131974248927|
|   1700.0| 1273.4154241645244|
|   1800.0| 1352.7482465277778|
|   1900.0| 1425.4899999999998|
|   2000.0| 1499.5295501730104|
|   2100.0|  1575.418603491272|
|   2200.0| 1649.5796498905909|
|   2300.0| 1721.9014420803783|
|   2400.0| 1802.1438375350142|
|   2500.0| 1874.5090566037732|
|   2600.0| 1953.1546927374302|
|   2700

Hospitals apply a fixed proportional discount relative to gross price.

Discount percentages are strikingly consistent across procedures.

Gross price is by far the strongest predictor of discounted price.

In [27]:
# Variation by Hospital (Price Level Differences)

df_dedup.groupBy("hospital_name") \
    .agg(F.avg("discounted_cash").alias("mean_cash"),
         F.stddev("discounted_cash").alias("std_cash")) \
    .orderBy("mean_cash", ascending=False) \
    .show()


+-----------------+------------------+------------------+
|    hospital_name|         mean_cash|          std_cash|
+-----------------+------------------+------------------+
|    massachusetts| 6647.069436090213|  97368.8106161954|
| newton-wellesley| 2699.292643643495| 5526.346063160736|
|            salem| 2670.610768469158|  5652.78172421528|
|nantucket cottage|1132.1979061014265| 3155.615867513764|
|           mclean|  591.059072745902|2358.0657944252443|
+-----------------+------------------+------------------+



Massachusetts General (the academic medical center) has much higher procedure complexity, causing extremely high-average cash prices and huge standard deviation.

Smaller or specialty hospitals (McLean = psychiatric, Nantucket = community) have much lower pricing windows.

In [28]:
# Variation by Procedure Type (code_2_type: CPT vs HCPCS)
df_dedup.groupBy("code_2_type") \
    .agg(F.avg("discounted_cash").alias("avg_cash")) \
    .orderBy("avg_cash", ascending=False) \
    .show()


+-----------+------------------+
|code_2_type|          avg_cash|
+-----------+------------------+
|    unknown|         10957.968|
|      hcpcs| 7731.615631265104|
|      local| 7612.551479628306|
|        cpt|2582.9620714925923|
+-----------+------------------+



HCPCS codes (J-codes, Q-codes, etc.) often represent drugs and injectable biologics, which are extremely expensive (hundreds of thousands).

CPT procedures (surgeries, imaging) are far less costly.

“unknown” may represent inpatient DRG-like or internal codes associated with very high charges.

In [29]:
# Inpatient vs Outpatient Price Comparison
df_dedup.groupBy("setting") \
    .agg(F.avg("discounted_cash").alias("avg_cash"),
         F.percentile_approx("discounted_cash", 0.5).alias("median_cash")) \
    .show()


+----------+------------------+-----------+
|   setting|          avg_cash|median_cash|
+----------+------------------+-----------+
|outpatient|3622.2649817889214|     563.25|
| inpatient| 3563.677979840757|     561.75|
+----------+------------------+-----------+



In [30]:
# Which Features Explain the Most Variance?

df_dedup.groupBy("code_2") \
    .agg(F.avg("discounted_cash").alias("avg_cash")) \
    .orderBy("avg_cash", ascending=False) \
    .show(20)


+--------+----------+
|  code_2|  avg_cash|
+--------+----------+
|89109999| 7012500.0|
|   Q2056|1664051.91|
|   Q2055| 1588675.5|
|   Q2054|1553832.94|
|   Q2041| 1472625.0|
|   Q2042|1456499.44|
|   Q2053| 1351500.0|
|   J9999|820781.255|
|   J2326| 452745.58|
|   J0225| 380431.31|
|27000895| 250218.75|
|81900003|  243673.5|
|   Q2043| 232948.87|
|27002207|  194400.0|
|   J9029|  191250.0|
|   J0224| 185987.44|
|   A9513|  178168.5|
|81900005|  170112.0|
|81100003| 168543.75|
|81200001|  163524.0|
+--------+----------+
only showing top 20 rows



Exploratory data analysis revealed a highly structured relationship between discounted cash price and gross charges. Binned averages showed a near-linear increase in discounted cash price with increasing gross charge, confirming a proportional pricing scheme commonly observed in hospital self-pay policies. This finding is supported by an almost perfect Pearson correlation coefficient (r ≈ 0.99999999) between the two variables, indicating that gross charge is the dominant determinant of discounted cash price. Additional grouping analyses demonstrated substantial variation across hospitals, procedure code types, and care settings. Hospitals with more complex caseloads exhibited markedly higher mean discounted prices, while HCPCS-coded procedures—typically representing drugs and biologics—displayed significantly elevated cost levels relative to CPT-coded services. These results underscore the importance of including gross price, hospital identity, procedure code, and setting as primary predictive features in the downstream machine learning model.

In [43]:
df_dedup.columns

['hospital_name',
 'code_2',
 'setting',
 'discounted_cash',
 'gross_price',
 'description',
 'code_1',
 'code_1_type',
 'code_2_type',
 'code_3']

### Data Preparation for Modeling

In [44]:
# handling missing values

# 1. Remove invalid target/gross rows
df_clean = df_dedup.filter(
    (F.col("discounted_cash").isNotNull()) &
    (F.col("discounted_cash") > 0) &
    (F.col("gross_price").isNotNull()) &
    (F.col("gross_price") > 0)
)

# 2. Replace missing textual and categorical fields
df_clean = df_clean.fillna({
    "description": "",
    "code_2": "NO_CODE_2",
    "code_2_type": "unknown",
    "code_1": "NONE",
    "code_1_type": "unknown",
    "code_3": "NO_CODE_3",
    "setting": "unknown_setting",
    "hospital_name": "unknown_hospital"
})



In [45]:
for c in df_clean.columns:
    print(c, df_clean.filter(F.col(c).isNull()).count())


hospital_name 0
code_2 0
setting 0
discounted_cash 0
gross_price 0
description 0
code_1 0
code_1_type 0
code_2_type 0
code_3 0


In [46]:
from pyspark.sql import functions as F

# Log-transform target and numeric predictor
df_model = (
    df_clean
    .withColumn("log_cash", F.log(F.col("discounted_cash")))
    .withColumn("log_gross", F.log(F.col("gross_price")))
)

# Train / Validation / Test split
train_df, val_df, test_df = df_model.randomSplit([0.7, 0.15, 0.15], seed=42)
print(train_df.count(), val_df.count(), test_df.count())


33251 7001 6972


In [47]:
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline


In [48]:
text_col = "description"

categorical_cols = [
    "hospital_name",
    "code_2",
    "setting",
    "code_1_type",
    "code_2_type"
]

numeric_cols = ["log_gross"]
target_col = "log_cash"


In [49]:
# description prep
tokenizer = Tokenizer(inputCol=text_col, outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashing_tf = HashingTF(inputCol="filtered", outputCol="tf_raw", numFeatures=20000)
idf = IDF(inputCol="tf_raw", outputCol="tfidf_features")


In [50]:
# Categorical variables - OHE
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"])
    for c in categorical_cols
]


In [51]:
# Numeric Variables - Standard Scaler

num_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="num_unscaled"
)

scaler = StandardScaler(
    inputCol="num_unscaled",
    outputCol="num_scaled",
    withMean=True,
    withStd=True
)


In [52]:
# Assembling final feature vect

feature_cols = ["tfidf_features"] + [f"{c}_vec" for c in categorical_cols] + ["num_scaled"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)


In [53]:
# Defining Linear Regression model

lr = LinearRegression(
    featuresCol="features",
    labelCol=target_col,
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)


In [54]:
# pipeline

pipeline = Pipeline(stages=
    [tokenizer, remover, hashing_tf, idf] +
    indexers + encoders +
    [num_assembler, scaler, assembler, lr]
)


In [55]:
# fitting pipeline

pipeline_model = pipeline.fit(train_df)

train_pred = pipeline_model.transform(train_df)
val_pred   = pipeline_model.transform(val_df)
test_pred  = pipeline_model.transform(test_df)


In [42]:
df_model.filter(F.col("log_gross").isNull()).count()


21

In [56]:
# evaluation

from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(df, label="log_cash"):
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction")

    rmse = evaluator.setMetricName("rmse").evaluate(df)
    mae  = evaluator.setMetricName("mae").evaluate(df)
    r2   = evaluator.setMetricName("r2").evaluate(df)

    # Calculate MAPE manually
    df_m = df.withColumn("ape", F.abs((F.exp(F.col("prediction")) - F.exp(F.col(label))) / F.exp(F.col(label))))
    mape = df_m.agg(F.mean("ape")).first()[0]

    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}


In [57]:
print("TRAIN:", evaluate(train_pred))
print("VALID:", evaluate(val_pred))
print("TEST:", evaluate(test_pred))


TRAIN: {'RMSE': 0.17508234667697603, 'MAE': 0.0823278982612729, 'R2': 0.9930655054521644, 'MAPE': 0.0968552143378268}
VALID: {'RMSE': 0.2811988415004916, 'MAE': 0.12550898219485113, 'R2': 0.9813201149550008, 'MAPE': 0.24203867231511486}
TEST: {'RMSE': 0.2732444927036042, 'MAE': 0.12514627550987256, 'R2': 0.982796110234425, 'MAPE': 0.23323983126728431}


In [60]:
# Storing clean data to parquet

# df_clean.write.mode("overwrite").parquet("/content/drive/My Drive/ProjectBigData/01 datasets/hospital_prices_clean/")


In [61]:
# # Single parquet file
# df_clean.coalesce(1).write.mode("overwrite").parquet(
#     "/content/drive/My Drive/ProjectBigData/05 artifacts/hospital_prices_clean_singlefile/"
# )


In [62]:
# Saving the pipeline

# pipeline_model.write().overwrite().save("/content/drive/My Drive/ProjectBigData/05 artifacts/hospital_price_pipeline")

## Tetsing on one data point

In [63]:
# Loading parquet files

from pyspark.ml.pipeline import PipelineModel

loaded_pipeline = PipelineModel.load("/content/drive/My Drive/ProjectBigData/05 artifacts/hospital_price_pipeline")


In [65]:
matched_description = "mri lower extremity joint w contrast"
code2 = "73723"              # CPT for MRI knee with contrast
code2_type = "cpt"
code1_type = "cdm"
code1 = "PX-31000022"
code3 = "0352"
setting = "outpatient"
hospital = "massachusetts"

In [66]:
import math

test_row = [{
    "description": "mri lower extremity joint w contrast",
    "hospital_name": "massachusetts",
    "code_2": "73723",
    "code_2_type": "cpt",
    "code_1": "PX-31000022",
    "code_1_type": "cdm",
    "code_3": "0352",
    "setting": "outpatient",
    "gross_price": 4530.00,
    "log_gross": math.log(4530.00)
}]

user_input_df = spark.createDataFrame(test_row)


In [68]:
pred = loaded_pipeline.transform(user_input_df)
pred.select("prediction").show()
pred.withColumn("pred_cash", F.exp("prediction")).select("pred_cash").show()

+-----------------+
|       prediction|
+-----------------+
|7.752202004535626|
+-----------------+

+----------------+
|       pred_cash|
+----------------+
|2326.69016017937|
+----------------+



In [69]:
# Testing all hospitals at once

hospitals = [
    ("massachusetts", 4530.00),
    ("newton-wellesley", 3825.00),
    ("salem", 3990.00),
    ("nantucket cottage", 2990.00),
    ("mclean", 1850.00)
]

input_rows = []

for hospital, gp in hospitals:
    input_rows.append({
        "description": "mri lower extremity joint w contrast",
        "hospital_name": hospital,
        "code_2": "73723",
        "code_2_type": "cpt",
        "code_1": "PX-31000022",
        "code_1_type": "cdm",
        "code_3": "0352",
        "setting": "outpatient",
        "gross_price": gp,
        "log_gross": math.log(gp)
    })

user_input_df = spark.createDataFrame(input_rows)

pred = loaded_pipeline.transform(user_input_df)
pred = pred.withColumn("pred_cash", F.exp("prediction"))

pred.select("hospital_name", "pred_cash").show()


+-----------------+------------------+
|    hospital_name|         pred_cash|
+-----------------+------------------+
|    massachusetts|  2326.69016017937|
| newton-wellesley| 1955.553579039184|
|            salem|2004.8728019213079|
|nantucket cottage|1635.3069896057073|
|           mclean| 928.7064137252742|
+-----------------+------------------+



# Creating Description embeddings

In [70]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [82]:
# df_clean = spark.read.parquet("/content/drive/My Drive/ProjectBigData/01 datasets/hospital_prices_clean/")

In [71]:
df_clean.show(5)

+-------------+------+---------+---------------+-----------+--------------------+-----------+-----------+-----------+------+
|hospital_name|code_2|  setting|discounted_cash|gross_price|         description|     code_1|code_1_type|code_2_type|code_3|
+-------------+------+---------+---------------+-----------+--------------------+-----------+-----------+-----------+------+
|massachusetts| 0001U|inpatient|         987.75|     1317.0|rbc dna hea 35 ag...|PX-30003410|        cdm|        cpt|  0300|
|massachusetts| 0008U|inpatient|          381.0|      508.0|hpylori detection...|PX-31001079|        cdm|        cpt|  0310|
|massachusetts| 0027U|inpatient|        1301.25|     1735.0|jak2 exons 12 - 1...|PX-31000942|        cdm|        cpt|  0310|
|massachusetts| 0031A|inpatient|          56.25|       75.0|imm admn sarscov2...|PX-77100033|        cdm|        cpt|  0771|
|massachusetts| 0035U|inpatient|         4723.5|     6298.0|neuro csf detcj p...|PX-31001015|        cdm|        cpt|  0310|


In [72]:
df_clean.select("description").distinct().count()

10144

In [83]:
w = Window.orderBy("hospital_name", "code_2", "setting")

df_aligned = df_clean.withColumn("seq_id", F.row_number().over(w) - 1)

In [84]:
# Extract row_id + description in same order - extract descriptions into pandas
pdf = df_aligned.orderBy("seq_id").select("seq_id", "description").toPandas()
descriptions = pdf["description"].tolist()


In [75]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [85]:
desc_emb = embed_model.encode(descriptions, batch_size=128, show_progress_bar=True)

Batches:   0%|          | 0/369 [00:00<?, ?it/s]

In [86]:
# # Saving embeddings

# np.save("/content/drive/My Drive/ProjectBigData/05 artifacts/desc_emb.npy", desc_emb)
# pdf.to_csv("/content/drive/My Drive/ProjectBigData/05 artifacts/seqid_to_description.csv", index=False)


description_embeddings.npy → embeddings matrix

rowid_description_map.csv → which embedding corresponds to which row_id

In [79]:
# Loading embeddings for prediction

import numpy as np
desc_emb = np.load("/content/drive/My Drive/ProjectBigData/05 artifacts/desc_emb.npy")

import pandas as pd
row_map = pd.read_csv("/content/drive/My Drive/ProjectBigData/05 artifacts/seqid_to_description.csv")


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import math
from pyspark.sql import Row
from pyspark.sql import functions as F

In [107]:
import numpy as np
import math
from sklearn.metrics.pairwise import cosine_similarity
from pyspark.sql import functions as F

# ------------------------------------------------------------------------
# ANATOMY MAPPING
# ------------------------------------------------------------------------
ANATOMY = {
    "knee": ["knee", "patellar", "patella", "lower extremity", "leg"],
    "hip": ["hip", "pelvis"],
    "spine": ["spine", "lumbar", "thoracic", "cervical"],
    "shoulder": ["shoulder", "rotator cuff"],
    "elbow": ["elbow"],
    "wrist": ["wrist"],
    "hand": ["hand"],
    "ankle": ["ankle"],
    "foot": ["foot"],
    "brain": ["brain", "head", "cranial"],
    "abdomen": ["abdomen", "abdominal"],
    "heart": ["heart", "cardiac"],
    "lung": ["lung", "chest", "thorax"]
}

# ------------------------------------------------------------------------
# Utility: detect anatomical region from user text
# ------------------------------------------------------------------------
def detect_anatomy(user_text):
    text = user_text.lower()
    for region, terms in ANATOMY.items():
        if any(t in text for t in terms):
            return region
    return None

# ------------------------------------------------------------------------
# Utility: check if description matches same anatomy
# ------------------------------------------------------------------------
def same_anatomy(desc, region):
    if region is None:
        return False
    desc = desc.lower()
    for term in ANATOMY[region]:
        if term in desc:
            return True
    return False

# ------------------------------------------------------------------------
# Main prediction function v4
# ------------------------------------------------------------------------
def predict_price_v4(
    user_text,
    df_aligned,
    df_clean,
    desc_emb,
    embed_model,
    spark,
    pipeline_model,
    top_k=15
):
    """
    Final enhanced prediction function:
    - semantic search
    - imaging-aware filtering
    - anatomy-aware filtering
    - outpatient prioritization AFTER anatomy check
    """

    # --------------------------------------------------------------------
    # STEP 1 — Embed the user text
    # --------------------------------------------------------------------
    user_emb = embed_model.encode([user_text])

    # --------------------------------------------------------------------
    # STEP 2 — Semantic similarity search
    # --------------------------------------------------------------------
    scores = cosine_similarity(user_emb, desc_emb)[0]
    top_indices = scores.argsort()[::-1][:top_k]

    # Build candidate list
    candidates = []
    for idx in top_indices:
        row = df_aligned.filter(F.col("seq_id") == int(idx)).first()
        if row:
            candidates.append({
                "seq_id": int(idx),
                "description": row.description,
                "similarity": float(scores[idx]),
                "code_2": row.code_2,
                "code_2_type": row.code_2_type,
                "code_1": row.code_1,
                "code_1_type": row.code_1_type,
                "code_3": row.code_3,
                "setting": row.setting
            })

    if len(candidates) == 0:
        raise ValueError("No semantic matches found.")

    # --------------------------------------------------------------------
    # STEP 3 — Imaging-aware filter (MRI/CT/X-ray → CPT 7xxxx only)
    # --------------------------------------------------------------------
    user_lower = user_text.lower()
    imaging_terms = ["mri", "ct", "scan", "xray", "x-ray", "x ray", "ultrasound"]

    user_wants_imaging = any(t in user_lower for t in imaging_terms)

    if user_wants_imaging:
        imaging_only = [c for c in candidates if str(c["code_2"]).startswith("7")]
        if len(imaging_only) > 0:
            candidates = imaging_only

    # --------------------------------------------------------------------
    # STEP 4 — Anatomy-aware prioritization
    # --------------------------------------------------------------------
    anatomy = detect_anatomy(user_text)

    if anatomy:
        anatomy_matches = [c for c in candidates if same_anatomy(c["description"], anatomy)]
        if len(anatomy_matches) > 0:
            candidates = anatomy_matches

    # --------------------------------------------------------------------
    # STEP 5 — Outpatient prioritization
    # --------------------------------------------------------------------
    outpatient = [c for c in candidates if c["setting"] and c["setting"].lower() == "outpatient"]

    if len(outpatient) > 0:
        selected = outpatient[0]
    else:
        selected = candidates[0]

    # Extract final chosen procedure
    proc_desc   = selected["description"]
    code_2      = selected["code_2"]
    code_2_type = selected["code_2_type"]
    code_1      = selected["code_1"]
    code_1_type = selected["code_1_type"]
    code_3      = selected["code_3"]
    setting     = selected["setting"]

    # --------------------------------------------------------------------
    # STEP 6 — Find hospitals that publish this CPT + setting
    # --------------------------------------------------------------------
    hospitals = (
        df_clean
        .filter(F.col("code_2") == code_2)
        .filter(F.col("setting") == setting)
        .select("hospital_name", "gross_price")
        .collect()
    )

    if len(hospitals) == 0:
        raise ValueError(f"No hospitals publish CPT={code_2}, setting={setting}")

    # --------------------------------------------------------------------
    # STEP 7 — Build model inputs
    # --------------------------------------------------------------------
    rows = []
    for h in hospitals:
        gp = float(h.gross_price)
        rows.append({
            "description": proc_desc,
            "hospital_name": h.hospital_name,
            "code_2": code_2,
            "code_2_type": code_2_type,
            "code_1": code_1,
            "code_1_type": code_1_type,
            "code_3": code_3,
            "setting": setting,
            "gross_price": gp,
            "log_gross": math.log(gp)
        })

    spark_df = spark.createDataFrame(rows)

    # --------------------------------------------------------------------
    # STEP 8 — Predict
    # --------------------------------------------------------------------
    pred_df = pipeline_model.transform(spark_df)
    pred_df = pred_df.withColumn("pred_cash", F.exp(F.col("prediction")))
    result = pred_df.select("hospital_name", "pred_cash").orderBy("hospital_name")

    return result, candidates, selected


In [108]:
result_df, matches, selected = predict_price_v4(
    "MRI scan knee with dye",
    df_aligned=df_aligned,
    df_clean=df_clean,
    desc_emb=desc_emb,
    embed_model=embed_model,
    spark=spark,
    pipeline_model=pipeline_model
)

result_df.show()
selected


+-----------------+------------------+
|    hospital_name|         pred_cash|
+-----------------+------------------+
|    massachusetts| 1019.239467682776|
|           mclean|176.31476547302586|
|nantucket cottage|1020.3903110349875|
| newton-wellesley| 993.1170705508217|
|            salem| 998.2415008839557|
+-----------------+------------------+



{'seq_id': 14046,
 'description': 'mri, joint of leg. combo',
 'similarity': 0.58927321434021,
 'code_2': '73723',
 'code_2_type': 'cpt',
 'code_1': 'PX-98301632',
 'code_1_type': 'cdm',
 'code_3': '0983',
 'setting': 'inpatient'}

In [109]:
result_df, matches, selected = predict_price_v4(
    "colonoscopy with biopsy removal",
    df_aligned=df_aligned,
    df_clean=df_clean,
    desc_emb=desc_emb,
    embed_model=embed_model,
    spark=spark,
    pipeline_model=pipeline_model
)

result_df.show()
selected


+-----------------+------------------+
|    hospital_name|         pred_cash|
+-----------------+------------------+
|    massachusetts|1969.1824738929135|
|nantucket cottage| 1602.495471335227|
| newton-wellesley| 1705.296906826042|
|            salem|1714.0961465687365|
+-----------------+------------------+



{'seq_id': 16855,
 'description': 'colonoscopy w/biopsy single/multiple',
 'similarity': 0.8290833830833435,
 'code_2': '45380',
 'code_2_type': 'cpt',
 'code_1': 'PX-98301073',
 'code_1_type': 'cdm',
 'code_3': '0983',
 'setting': 'outpatient'}

In [113]:
result_df, matches, selected = predict_price_v4(
    "knee replacement surgery",
    df_aligned=df_aligned,
    df_clean=df_clean,
    desc_emb=desc_emb,
    embed_model=embed_model,
    spark=spark,
    pipeline_model=pipeline_model
)

result_df.show()
selected


+-------------+-----------------+
|hospital_name|        pred_cash|
+-------------+-----------------+
|massachusetts|8270.249039634522|
+-------------+-----------------+



{'seq_id': 1745,
 'description': 'revise knee joint replace,all parts',
 'similarity': 0.6786899566650391,
 'code_2': '27487',
 'code_2_type': 'cpt',
 'code_1': 'PX-76002363',
 'code_1_type': 'cdm',
 'code_3': '0761',
 'setting': 'outpatient'}

In [114]:
# base_path = "/content/drive/My Drive/ProjectBigData/05 artifacts/"

# df_clean.write.mode("overwrite").parquet(base_path + "hospital_prices_clean/")
# df_aligned.write.mode("overwrite").parquet(base_path + "hospital_prices_aligned/")
# import numpy as np
# np.save(base_path + "desc_emb.npy", desc_emb)
# (
#     df_aligned
#     .select("seq_id", "description")
#     .orderBy("seq_id")
#     .toPandas()
#     .to_csv(base_path + "seqid_to_description.csv", index=False)
# )
# pipeline_model.write().overwrite().save(base_path + "hospital_price_pipeline")


In [115]:
# embed_model.save(base_path + "sentence_transformer_model")
